<a href='https://www.kaggle.com/datasets/yash9439/loan-prediction/data'>Dataset</a>

In [252]:
#%pip install xgboost

In [253]:
import os
import zipfile
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split 
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import SMOTE
import numpy as np

In [254]:
cw = os.getcwd()
models = cw+'/models'
extracted_data = cw+'/extracted_data'
zip_path = cw+'/archive.zip'

folders = [models, extracted_data]

if not os.path.exists(extracted_data) or not os.path.exists(models):
    for folder in folders:
        os.makedirs(folder, exist_ok=True)

try:
    with zipfile.ZipFile(zip_path, mode='r') as file:
        file.extractall(path=extracted_data)
except zipfile.BadZipFile as e:
    print(f"Error extracting zipfile: {e}")

In [255]:
df = pd.read_csv('./extracted_data/Loan_Train.txt')

In [256]:
df.dropna(inplace=True)
df.isna().sum()

Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

In [257]:
le = LabelEncoder()
df["Loan_Status"] = le.fit_transform(df["Loan_Status"])
df["Married"] = le.fit_transform(df["Married"])
df["Education"] = le.fit_transform(df["Education"])
df["Self_Employed"] = le.fit_transform(df["Self_Employed"])
df["Gender"] = le.fit_transform(df["Gender"])

df = pd.get_dummies(data=df, columns=["Property_Area"], drop_first=True)


In [258]:
df["Dependents"] = df["Dependents"].apply(lambda x: 3 if x == '3+' else x)
df["Monthly_Payments"] = df["LoanAmount"] / df["Loan_Amount_Term"]
df["TotalIncome"] = df["ApplicantIncome"] + df["CoapplicantIncome"]
df["Debt_to_Income"] = df["LoanAmount"] / (df["TotalIncome"] + 1)
df["LoanAmount_to_Income"] = df["LoanAmount"] / (df["ApplicantIncome"] + 1)

In [259]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status,Property_Area_Semiurban,Property_Area_Urban,Monthly_Payments,TotalIncome,Debt_to_Income,LoanAmount_to_Income
1,LP001003,1,1,1,0,0,4583,1508.0,128.0,360.0,1.0,0,False,False,0.355556,6091.0,0.021011,0.027923
2,LP001005,1,1,0,0,1,3000,0.0,66.0,360.0,1.0,1,False,True,0.183333,3000.0,0.021993,0.021993
3,LP001006,1,1,0,1,0,2583,2358.0,120.0,360.0,1.0,1,False,True,0.333333,4941.0,0.024282,0.046440
4,LP001008,1,0,0,0,0,6000,0.0,141.0,360.0,1.0,1,False,True,0.391667,6000.0,0.023496,0.023496
5,LP001011,1,1,2,0,1,5417,4196.0,267.0,360.0,1.0,1,False,True,0.741667,9613.0,0.027772,0.049280


In [263]:
y = df["Loan_Status"]
X = df.drop(columns=['Loan_ID', 'Loan_Status', 'ApplicantIncome', 'CoapplicantIncome'])
bool_cols = ['Property_Area_Semiurban', 'Property_Area_Urban']
X[bool_cols] = X[bool_cols].astype(int)

y.value_counts()

Loan_Status
1    332
0    148
Name: count, dtype: int64

In [264]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [265]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression()),
])

param_grid = [
    {
        'clf': [LogisticRegression()],
        'clf__max_iter': [100, 500, 1000],
        'clf__solver': ['lbfgs', 'liblinear'],
        'clf__C': [0.1, 1, 10, 30],
        'clf__class_weight': ['balanced', {0:2, 1:1}, {0:3, 1:1}]
    },
    {
        'clf': [SVC(random_state=42, probability=True)],
        'clf__kernel': ['rbf', 'linear'],
        'clf__gamma': ['scale', 0.1, 0.01],
        'clf__C': [0.1, 1, 10, 20],
        'clf__class_weight': ['balanced', {0:2, 1:1}, {0:3, 1:1}]
    },
    {
        'clf': [RandomForestClassifier(random_state=42)],
        'clf__max_depth': [3, 5, 15],
        'clf__n_estimators': [100, 200],
        'clf__min_samples_leaf': [1, 5, 10],
        'clf__class_weight': ['balanced', {0:2, 1:1}, {0:3, 1:1}]
    },
    {
    'clf': [XGBClassifier(random_state=42, eval_metric='logloss')],
    'clf__max_depth': [3, 5, 7],
    'clf__n_estimators': [100, 200],
    'clf__learning_rate': [0.01, 0.05, 0.1],
    'clf__scale_pos_weight': [1, 2, 3]
    }
]

grid_search = GridSearchCV(
    pipe, param_grid,
    cv=5,
    scoring='f1_macro'
)

In [ ]:
grid_search.fit(X_train_resampled, y_train_resampled)
print('Best model: ', grid_search.best_estimator_)
print('Best params: ', grid_search.best_params_)
print('Best score: ', grid_search.best_score_)

Best model:  Pipeline(steps=[('scaler', StandardScaler()),
                ('clf',
                 RandomForestClassifier(class_weight='balanced', max_depth=5,
                                        random_state=42))])
Best params:  {'clf': RandomForestClassifier(random_state=42), 'clf__class_weight': 'balanced', 'clf__max_depth': 5, 'clf__min_samples_leaf': 1, 'clf__n_estimators': 100}
Best score:  0.8349816944731435


In [ ]:
results_df = pd.DataFrame(grid_search.cv_results_)
ordered_results = results_df.sort_values('rank_test_score', ascending=True)
ordered_results.head(3)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_clf,param_clf__C,param_clf__class_weight,param_clf__max_iter,param_clf__solver,param_clf__gamma,...,param_clf__scale_pos_weight,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
150,0.142879,0.003094,0.010427,0.000283,RandomForestClassifier(random_state=42),NaN,balanced,NaN,NaN,NaN,...,NaN,{'clf': RandomForestClassifier(random_state=42...,0.763393,0.686969,0.877348,0.914255,0.932944,0.834982,0.094560,1
151,0.285512,0.001984,0.016856,0.000377,RandomForestClassifier(random_state=42),NaN,balanced,NaN,NaN,NaN,...,NaN,{'clf': RandomForestClassifier(random_state=42...,0.763393,0.675961,0.886792,0.914255,0.933115,0.834703,0.099006,2
219,0.058595,0.000737,0.005858,0.000050,"XGBClassifier(base_score=None, booster=None, c...",NaN,NaN,NaN,NaN,NaN,...,1.0,"{'clf': XGBClassifier(base_score=None, booster...",0.736828,0.708578,0.858478,0.914255,0.942810,0.832190,0.093847,3


In [ ]:
grid_search.best_params_

{'clf': RandomForestClassifier(random_state=42),
 'clf__class_weight': 'balanced',
 'clf__max_depth': 5,
 'clf__min_samples_leaf': 1,
 'clf__n_estimators': 100}

In [270]:
model = RandomForestClassifier(
    class_weight={0: 3, 1: 1},
    max_depth=15,
    min_samples_leaf=1,
    n_estimators=200,
    random_state=42
)
model.fit(X_train_resampled, y_train_resampled)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [271]:
proba = model.predict_proba(X_test)

threshold_params = np.arange(0.1, 0.9, 0.05)

scores = [f1_score(y_test, (proba[:, 1] >= t).astype(int), average='macro') 
          for t in threshold_params]

best_threshold = threshold_params[np.argmax(scores)]
print(f"Best Threshold: {best_threshold}\nScore: {max(scores):.3f}")

preds = (proba[:, 1] >= best_threshold).astype(int)

Best Threshold: 0.45000000000000007
Score: 0.771


In [272]:
importances = pd.Series(model.feature_importances_, index=X.columns)
print(importances.sort_values(ascending=False))

Credit_History             0.193355
TotalIncome                0.121835
Debt_to_Income             0.121510
LoanAmount                 0.115205
LoanAmount_to_Income       0.113667
Monthly_Payments           0.106335
Property_Area_Semiurban    0.052442
Dependents                 0.044176
Married                    0.034210
Loan_Amount_Term           0.028605
Property_Area_Urban        0.020652
Gender                     0.020209
Education                  0.014008
Self_Employed              0.013792
dtype: float64


In [273]:
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.83      0.54      0.65        28
           1       0.83      0.96      0.89        68

    accuracy                           0.83        96
   macro avg       0.83      0.75      0.77        96
weighted avg       0.83      0.83      0.82        96



In [274]:
print(confusion_matrix(y_test, preds))

[[15 13]
 [ 3 65]]
